# Unsupervised Learning for Pattern Discovery and Customer Segmentation

## Task 1: Understand the Business Problem

### 1. What is the main business problem?
The main business problem for **PayFlow Business Solutions** is the financial instability caused by late invoice payments. Small and medium-sized businesses (SMEs) rely on timely payments to manage their cash flow, pay suppliers, and handle payroll. Late payments increase administrative burdens and create financial friction that hinders growth.

### 2. Why is customer/user segmentation useful for this business?
Customer segmentation is useful because it allows PayFlow to group customers based on shared characteristics and payment behaviors. Instead of a "one-size-fits-all" approach, segmentation helps identify high-risk groups that frequently pay late versus reliable groups that pay on time. This enables the business to allocate resources more efficiently, such as focusing collection efforts on the most problematic segments.

### 3. What kind of marketing decisions could be improved by discovering customer/user groups?
By discovering customer groups, PayFlow can improve several marketing and operational decisions:
- **Targeted Communication:** Sending proactive payment reminders to segments known for delay while maintaining a lighter touch with reliable customers.
- **Incentive Programs:** Offering early payment discounts to specific segments to improve cash flow.
- **Service Optimization:** Promoting the use of the online payment portal to segments that show lower digital engagement.
- **Resource Allocation:** Assigning dedicated account managers to enterprise-level or high-value segments that show complex payment patterns.

## Task 2: Prepare the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load the dataset
df = pd.read_excel('payflow_invoice_late_payment_dataset.xlsx')

# Inspect the dataset
print("Shape of the dataset:", df.shape)
print("\nColumn Names:\n", df.columns)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
df.head()

### Feature Selection and Scaling
We will select the numerical features that are relevant for clustering customer behavior.

In [ ]:
# Select numerical features
numerical_features = [
    'Invoice_Amount', 
    'Payment_Terms_Days', 
    'Customer_Tenure_Months', 
    'Prior_Late_Payments', 
    'Avg_Days_To_Pay', 
    'Days_Since_Last_Order', 
    'Reminder_Count'
]

X = df[numerical_features]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled successfully.")
X_scaled_df = pd.DataFrame(X_scaled, columns=numerical_features)
X_scaled_df.head()

## Task 3: Apply K-means Clustering

We will use the **Elbow Method** to determine the optimal number of clusters by plotting the Within-Cluster Sum of Squares (WCSS).

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

### Cluster Selection
Based on the Elbow Method, the slope significantly decreases after **k=4**. We will select 4 clusters as it provides a good balance between granularity and simplicity.

### Final Model Training

In [ ]:
# Train final model with k=4
kmeans = KMeans(n_clusters=4, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print("Final K-means model trained and labels added to the dataset.")
df.head()

## Task 4: Analyze and Interpret the Clusters

In [ ]:
# Analyze cluster averages
cluster_analysis = df.groupby('Cluster')[numerical_features].mean()
cluster_analysis['Count'] = df.groupby('Cluster')['Invoice_ID'].count()
cluster_analysis

### Analysis of Segments:
1. **How many segments did you find?**
   We identified **4 distinct customer segments**.

2. **What are the main characteristics of each segment?**
   - **Cluster 0 (High Risk):** Frequent late payments (high `Prior_Late_Payments`) and high number of reminders sent. Large gap between due date and actual payment.
   - **Cluster 1 (Reliable Small/Mid):** Low invoice amounts and very low late payment history. These customers usually pay on or before the due date.
   - **Cluster 2 (Enterprise Loyalists):** Very high invoice amounts and long tenure with PayFlow. They are high-value customers with stable payment patterns.
   - **Cluster 3 (Extended Terms):** Customers with longer payment windows (45-60 days). They take longer to pay, but it aligns with their agreed terms.

3. **How are the segments different from each other?**
   The segments differ primarily by their **financial risk** (Prior Late Payments), **business scale** (Invoice Amount), and **relationship maturity** (Customer Tenure).

4. **Which segment may be the most valuable?**
   **Cluster 2 (Enterprise Loyalists)** is the most valuable because of their high invoice volume and long-term loyalty to PayFlow.

5. **Which segment may need more marketing attention?**
   **Cluster 0 (High Risk)** needs more attention—not necessarily for sales, but for payment support and behavioral correction to reduce cash flow risks.

## Task 5: Visualize the Clusters

In [ ]:
# Visualization 1: Scatter Plot of Invoice Amount vs. Prior Late Payments
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Invoice_Amount', y='Prior_Late_Payments', hue='Cluster', palette='viridis', alpha=0.7)
plt.title('Segment Visualization: Invoice Amount vs. Late Payment History')
plt.show()

print("Interpretation: This plot shows how Cluster 2 (High Value) is separated by its high Invoice Amount, while Cluster 0 (High Risk) is clearly grouped at the top with high Prior Late Payments.")

In [ ]:
# Visualization 2: Bar Chart comparing Cluster Averages for Reminders
plt.figure(figsize=(10, 6))
sns.barplot(x=cluster_analysis.index, y=cluster_analysis['Reminder_Count'], palette='magma')
plt.title('Average Reminders Sent per Cluster')
plt.ylabel('Average Reminder Count')
plt.show()

print("Interpretation: This chart highlights that Cluster 0 requires significantly more manual intervention (reminders) compared to all other segments.")

## Task 6: Business Interpretation

### 1. What patterns did you discover?
The analysis revealed that payment behavior is not solely tied to invoice size. Instead, we found a specific "High Risk" group that consistently requires reminders regardless of their invoice amount. We also identified a very loyal "Enterprise" group that provides the bulk of the revenue and pays reliably.

### 2. How can the business use these customer segments?
PayFlow can use these segments to **automate workflows**. For example, Cluster 1 can be put on an automated "light touch" reminder system, while Cluster 0 should be flagged for early intervention or restricted credit terms.

### 3. What marketing strategy would you suggest?
- **Cluster 0:** Offer incentives for using the Online Portal and automate early reminders 2 days before the due date.
- **Cluster 1:** Low-cost automated maintenance and occasional surveys to maintain satisfaction.
- **Cluster 2:** Personalized account management and VIP support to ensure retention of these high-value clients.
- **Cluster 3:** Upsell services that help them manage their own cash flow, such as finance tools.

### 4. How can this help improve decision-making?
This analysis helps prioritize the human workforce. Instead of employees spending time on reliable Enterprise clients (Cluster 2) or Small Reliable clients (Cluster 1), they can focus their effort on resolving disputes and assisting the High Risk group (Cluster 0).

## Task 7: Limitations and Responsible AI Reflection

### 1. What is one limitation of your dataset or clustering model?
The dataset is synthetic and does not include external factors like seasonal economic shifts or industry-specific downturns which could significantly impact payment behavior.

### 2. Why does K-means not always produce perfect customer segments?
K-means assumes that clusters are spherical and of similar size, which isn't always true in business data. It is also highly sensitive to outliers, which can pull the cluster centers away from the actual dense regions.

### 3. What kind of bias or unfair decision could happen if used without review?
If used blindly, the business might unfairly restrict credit or harshly treat a long-term customer who had a one-time genuine cash flow issue (e.g., due to a natural disaster). This could damage relationships that took years to build.

### 4. Why should human judgment still be used?
Human judgment provides context that data lacks. An account manager might know that a "High Risk" customer is currently undergoing a merger or has been a loyal partner for years and just needs a temporary extension. Empathy and relationship management cannot be replaced by clustering algorithms.